# Nonlinear conditional-independence sensitivity

Compare PCMCI+ graph-instability curves under linear partial correlation and a dependency-light nonlinear residual test. GPDC and CMIknn are attempted and explicitly recorded as skipped when their optional dependencies are unavailable.

In [ ]:
from pathlib import Path
import json, platform, subprocess, sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'markovianity_diagnostic').exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('project root not found')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from markovianity_diagnostic.experiments.graph_metrics import compute_graph_instability
from markovianity_diagnostic.experiments.nonlinear_tests import run_ci_test_comparison
from markovianity_diagnostic.experiments.simulations import scenario_order1_unconfounded, scenario_latent_common_driver


In [ ]:
OUT = ROOT / 'outputs' / 'comparisons' / 'nonlinear_ci_tests'
OUT.mkdir(parents=True, exist_ok=True)
P_VALUES = [1, 2, 3]
CI_TESTS = ['parcorr', 'nonlinear-residual', 'gpdc', 'cmiknn']
SCENARIOS = {
    'order1_unconfounded': scenario_order1_unconfounded(T=500, d=5, seed=42),
    'latent_common_driver': scenario_latent_common_driver(T=500, d=5, seed=43),
}

rows, runtimes = [], []
for scenario_name, scenario in SCENARIOS.items():
    runs = run_ci_test_comparison(scenario.X, CI_TESTS, P_VALUES)
    for test_name, run in runs.items():
        runtimes.append({'scenario': scenario_name, 'ci_test': test_name, 'status': run.status, 'runtime_seconds': run.runtime_seconds, 'error': run.error})
        if run.status == 'ok':
            for depth, value in compute_graph_instability(run.adjacency).items():
                rows.append({'scenario': scenario_name, 'ci_test': test_name, 'depth': depth, 'D_p': value})

results = pd.DataFrame(rows)
runtime = pd.DataFrame(runtimes)
results.to_csv(OUT / 'results.csv', index=False)
runtime.to_csv(OUT / 'runtime.csv', index=False)
assert {'parcorr', 'nonlinear-residual'}.issubset(set(runtime.query("status == 'ok'")['ci_test']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for axis, (scenario_name, frame) in zip(axes, results.groupby('scenario')):
    for test_name, curve in frame.groupby('ci_test'):
        axis.plot(curve['depth'], curve['D_p'], marker='o', label=test_name)
    axis.set(title=scenario_name, xlabel='conditioning depth', ylabel='D_p')
    axis.legend()
fig.tight_layout()
figure_path = OUT / 'ci_test_instability.png'
fig.savefig(figure_path, dpi=300)
plt.close(fig)

commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip() or 'unknown'
manifest = {
    'analysis': 'nonlinear_ci_tests',
    'status': 'complete',
    'git_commit': commit,
    'input_paths': [],
    'output_paths': [str(OUT / 'results.csv'), str(OUT / 'runtime.csv'), str(figure_path)],
    'method': 'PCMCI+',
    'method_params': {'ci_tests': CI_TESTS, 'p_values': P_VALUES, 'T': 500, 'd': 5},
    'p_values': P_VALUES,
    'random_seed': 42,
    'software_versions': {'python': platform.python_version()},
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
manifest
